# 🔋 Lithium Sector Scoreboard Analysis (Enhanced)
## RK Equity Lithium Scoreboard — January 31, 2026

Comprehensive comparative analysis of **80+ lithium companies** enriched with:
- **yfinance** — Live price history, fundamentals, financial statements
- **Technical Analysis** — EMA, RSI, MACD, Bollinger Bands
- **Fundamental Analysis** — P/E, P/B, EV/EBITDA, margins, ROE, balance sheet health
- **Macro Context** — Lithium prices, EV demand, interest rates (FRED/IEA)
- **SEC EDGAR** — Filing links & insider data for US-listed companies

*⚠️ NOT INVESTMENT ADVICE. DO YOUR OWN RESEARCH.*


In [1]:
# ═══════════════════════════════════════════════════════════════
#  SETUP, IMPORTS & CONFIGURATION
# ═══════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
import yfinance as yf
import time
import warnings
from datetime import datetime, timedelta
warnings.filterwarnings("ignore")

# ── CONFIG ───────────────────────────────────────────────────
SCOREBOARD_DATE = "January 31, 2026"
W_1M, W_3M, W_YTD = 0.40, 0.35, 0.25       # Momentum weights
TIER_MEGA, TIER_LARGE, TIER_MID = 10_000, 1_000, 100

# Technical params
EMA_SHORT, EMA_MED, EMA_LONG = 25, 50, 200
RSI_PERIOD, BB_PERIOD, BB_STD = 14, 20, 2
MACD_FAST, MACD_SLOW, MACD_SIGNAL = 12, 26, 9

CATEGORY_COLORS = {
    "Chemical Producers":      "#1f77b4",
    "Spodumene Producers":     "#ff7f0e",
    "Alt Investment Vehicles":  "#2ca02c",
    "Hard-Rock Emerging":      "#d62728",
    "Sedimentary":             "#9467bd",
    "DLE Brine":               "#8c564b",
    "Salar Brine":             "#e377c2",
}

# ── UTILITY FUNCTIONS ────────────────────────────────────────
def safe_get(d, key, default=np.nan):
    if d is None: return default
    val = d.get(key)
    return val if val is not None else default

def fetch_yf_info(ticker, delay=0.3):
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        time.sleep(delay)
        if info and info.get("regularMarketPrice") is not None:
            return info
        if info and info.get("currentPrice") is not None:
            return info
        return None
    except Exception:
        return None

def fetch_yf_history(ticker, period="2y", delay=0.3):
    try:
        stock = yf.Ticker(ticker)
        hist = stock.history(period=period)
        time.sleep(delay)
        if hist is not None and len(hist) > 20:
            return hist
        return None
    except Exception:
        return None

def calculate_rsi(series, period=14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def calculate_technical_indicators(hist_df):
    df = hist_df.copy()
    c = df["Close"]
    df["EMA_25"]  = c.ewm(span=EMA_SHORT, adjust=False).mean()
    df["EMA_50"]  = c.ewm(span=EMA_MED, adjust=False).mean()
    df["EMA_200"] = c.ewm(span=EMA_LONG, adjust=False).mean()
    ema_f = c.ewm(span=MACD_FAST, adjust=False).mean()
    ema_s = c.ewm(span=MACD_SLOW, adjust=False).mean()
    df["MACD"] = ema_f - ema_s
    df["MACD_Signal"] = df["MACD"].ewm(span=MACD_SIGNAL, adjust=False).mean()
    df["MACD_Hist"] = df["MACD"] - df["MACD_Signal"]
    df["BB_Mid"] = c.rolling(BB_PERIOD).mean()
    bb_std = c.rolling(BB_PERIOD).std()
    df["BB_Upper"] = df["BB_Mid"] + BB_STD * bb_std
    df["BB_Lower"] = df["BB_Mid"] - BB_STD * bb_std
    df["RSI"] = calculate_rsi(c, RSI_PERIOD)
    df["Vol_MA20"] = df["Volume"].rolling(20).mean()
    return df

def get_technical_signal(row):
    signals = {}
    price = row.get("Close", np.nan)
    ema200 = row.get("EMA_200", np.nan)
    ema50 = row.get("EMA_50", np.nan)
    rsi = row.get("RSI", np.nan)
    macd = row.get("MACD", np.nan)
    macd_sig = row.get("MACD_Signal", np.nan)
    bb_u = row.get("BB_Upper", np.nan)
    bb_l = row.get("BB_Lower", np.nan)

    signals["trend"] = ("Bullish" if price > ema200 else "Bearish") if pd.notna(price) and pd.notna(ema200) else "N/A"
    signals["ema_cross"] = ("Golden Cross" if ema50 > ema200 else "Death Cross") if pd.notna(ema50) and pd.notna(ema200) else "N/A"

    if pd.notna(rsi):
        signals["rsi"] = "Overbought" if rsi > 70 else ("Oversold" if rsi < 30 else "Neutral")
        signals["rsi_value"] = round(rsi, 1)
    else:
        signals["rsi"], signals["rsi_value"] = "N/A", np.nan

    signals["macd"] = ("Bullish" if macd > macd_sig else "Bearish") if pd.notna(macd) and pd.notna(macd_sig) else "N/A"

    if pd.notna(price) and pd.notna(bb_u) and pd.notna(bb_l) and (bb_u - bb_l) > 0:
        bb_pct = (price - bb_l) / (bb_u - bb_l)
        signals["bollinger"] = "Near Upper" if bb_pct > 0.8 else ("Near Lower" if bb_pct < 0.2 else "Mid-Band")
    else:
        signals["bollinger"] = "N/A"

    score = 0
    for sig, bull, bear in [("trend","Bullish","Bearish"),("ema_cross","Golden Cross","Death Cross"),
                             ("macd","Bullish","Bearish")]:
        if signals[sig] == bull: score += 1
        elif signals[sig] == bear: score -= 1
    if signals["rsi"] == "Oversold": score += 1
    elif signals["rsi"] == "Overbought": score -= 1
    if signals["bollinger"] == "Near Lower": score += 1
    elif signals["bollinger"] == "Near Upper": score -= 1
    signals["tech_score"] = score
    return signals

print(f"✅ Setup complete — {SCOREBOARD_DATE}")


✅ Setup complete — January 31, 2026


# 📋 Section 1: Scoreboard Data
All companies extracted from the RK Equity Lithium Scoreboard image.

In [2]:
# ═══════════════════════════════════════════════════════════════
#  RAW DATA — RK Equity Lithium Scoreboard, January 31, 2026
# ═══════════════════════════════════════════════════════════════
data = [
    # (company, ticker, currency, share_price, market_cap_usdm, return_1m, return_3m, return_ytd, location, category)
    # ── CHEMICAL PRODUCERS ────────────────────────────────────
    ("Qinghai Salt Lake Industry","000792.SZ","¥",32.78,24210,0.18,0.30,0.06,"China","Chemical Producers"),
    ("SQM","SQM","$",76.84,21948,0.12,0.57,0.10,"Global","Chemical Producers"),
    ("Ganfeng","1772.HK","$",60.50,20754,0.11,0.12,0.12,"Global","Chemical Producers"),
    ("Albemarle","ALB","$",170.63,20083,0.21,0.74,0.19,"Global","Chemical Producers"),
    ("Tianqi Lithium","002466.SZ","¥",54.55,12496,-0.03,-0.02,-0.04,"Global","Chemical Producers"),
    ("Chengxin Lithium","002240.SZ","¥",35.55,4542,0.06,0.44,-0.03,"Global","Chemical Producers"),
    ("Sichuan Yahua","002497.SZ","¥",24.70,3973,0.00,0.28,-0.02,"Global","Chemical Producers"),
    ("Jiangsu Dingsheng","603876.SS","¥",17.09,2217,0.15,0.47,0.14,"China","Chemical Producers"),
    ("YOUNGY","002192.SZ","¥",54.63,1980,0.05,0.13,0.00,"China","Chemical Producers"),
    ("Yibin Tianyuan","002386.SZ","¥",6.05,1099,0.09,0.04,0.11,"China","Chemical Producers"),
    ("Lithium Argentina","LAR","$",6.73,1093,0.21,0.60,0.13,"Argentina","Chemical Producers"),
    # ── SPODUMENE PRODUCERS ───────────────────────────────────
    ("PLS","PLS.AX","$",4.29,9023,0.03,0.30,0.00,"Australia, WA","Spodumene Producers"),
    ("Mineral Resources","MIN.AX","$",57.15,7375,0.05,0.19,0.03,"Australia, WA","Spodumene Producers"),
    ("IGO","IGO.AX","$",8.32,4114,0.04,0.50,0.01,"Australia, WA","Spodumene Producers"),
    ("Liontown Resources","LTR.AX","$",1.86,3561,0.16,0.62,0.15,"Australia, WA","Spodumene Producers"),
    ("AMG Critical Minerals","AMG.AS","€",36.12,1366,0.27,0.25,0.22,"Brazil","Spodumene Producers"),
    ("Sigma Lithium","SGML","$",10.78,1201,-0.18,0.66,-0.24,"Brazil","Spodumene Producers"),
    ("Elevra Lithium","ELV.AX","$",6.70,741,-0.18,0.72,-0.15,"Canada, Quebec","Spodumene Producers"),
    ("Kodal Minerals","KOD.L","£",0.48,133,0.48,0.63,0.58,"Mali","Spodumene Producers"),
    # ── ALT INVESTMENT VEHICLES ───────────────────────────────
    ("Global X Lithium & Battery Tech ETF","LIT","$",69.95,1480,0.08,0.12,0.06,"N/A","Alt Investment Vehicles"),
    ("Lithium Royalty Corp","LIRC.TO","$",10.04,403,0.06,0.40,0.04,"N/A","Alt Investment Vehicles"),
    ("Sprott Lithium Miners ETF","LITP","$",12.95,41,0.07,0.25,0.03,"N/A","Alt Investment Vehicles"),
    # ── HARD-ROCK EMERGING ────────────────────────────────────
    ("PMET Resources","PMET.TO","$",6.17,732,0.13,0.75,0.11,"Canada, Quebec","Hard-Rock Emerging"),
    ("Core Lithium","CXO.AX","$",0.24,408,-0.13,0.96,-0.15,"Australia, NT","Hard-Rock Emerging"),
    ("Q2 Metals Corp.","QTWO.V","$",2.30,329,0.20,1.47,0.17,"Canada, Quebec","Hard-Rock Emerging"),
    ("Wildcat Resources","WC8.AX","$",0.37,323,0.00,0.70,-0.03,"Australia, WA","Hard-Rock Emerging"),
    ("European Lithium","EUR.AX","$",0.24,267,0.47,0.15,0.52,"Austria","Hard-Rock Emerging"),
    ("Li-FT Power Ltd.","LIFT.V","$",6.41,233,0.44,0.95,0.46,"Canada, NWT","Hard-Rock Emerging"),
    ("Savannah Resources","SAV.L","£",5.20,183,0.40,0.33,0.38,"Portugal","Hard-Rock Emerging"),
    ("Frontier Lithium","FL.V","$",0.98,165,0.40,0.31,0.40,"Canada, Ontario","Hard-Rock Emerging"),
    ("Atlas Lithium","ATLX","$",5.03,134,0.19,-0.06,0.15,"Brazil","Hard-Rock Emerging"),
    ("Atlantic Lithium","ALL.L","£",12.90,131,0.24,0.44,0.21,"Ghana","Hard-Rock Emerging"),
    ("Lithium Ionic Corp.","LTH.V","$",1.02,130,-0.02,0.44,-0.03,"Brazil","Hard-Rock Emerging"),
    ("Delta Lithium","DLI.AX","$",0.25,117,0.11,0.43,0.11,"Australia, WA","Hard-Rock Emerging"),
    ("Andrada Mining","ATM.L","£",4.20,111,0.21,0.37,0.19,"Namibia","Hard-Rock Emerging"),
    ("Winsome Resources","WR1.AX","$",0.59,93,0.30,1.93,0.29,"Canada, Quebec","Hard-Rock Emerging"),
    ("Rock Tech Lithium","RCK.V","$",1.04,88,0.44,0.21,0.37,"Canada, Ontario","Hard-Rock Emerging"),
    ("Global Lithium Resources","GL1.AX","$",0.50,85,-0.22,-0.06,-0.18,"Australia, WA","Hard-Rock Emerging"),
    ("Critical Elements Corp","CRE.V","$",0.47,80,0.12,0.07,0.09,"Canada, Quebec","Hard-Rock Emerging"),
    ("MAX Power Mining","MAXXF","$",0.80,76,0.72,0.47,0.73,"USA, Arizona","Hard-Rock Emerging"),
    ("Flagship Minerals","FLG.AX","$",0.24,49,0.17,0.23,0.00,"Thailand","Hard-Rock Emerging"),
    ("Zinnwald Lithium","ZNWD.L","£",7.15,46,0.24,0.11,0.15,"Germany","Hard-Rock Emerging"),
    ("European Metals Holdings","EMH.AX","$",0.33,45,-0.08,0.27,-0.08,"Czechia","Hard-Rock Emerging"),
    ("Brunswick Exploration","BRW.V","$",0.24,43,0.09,1.00,0.20,"Canada, Quebec","Hard-Rock Emerging"),
    ("Midland Exploration","MD.V","$",0.51,42,0.11,0.09,0.11,"Canada, Quebec","Hard-Rock Emerging"),
    ("MetalsTech Limited","MTC.AX","$",0.25,39,-0.07,-0.06,-0.07,"Canada, Quebec","Hard-Rock Emerging"),
    ("Ore Resources","OR3.AX","$",0.07,34,0.12,-0.04,0.12,"Australia, WA","Hard-Rock Emerging"),
    ("Iris Metals","IR1.AX","$",0.20,32,0.15,-0.35,-0.07,"USA, SD","Hard-Rock Emerging"),
    ("Mont Royal Resources","MRZ.AX","$",0.23,29,-0.19,-0.18,-0.18,"Canada, Quebec","Hard-Rock Emerging"),
    ("Battery Age Minerals","BM8.AX","$",0.17,24,0.32,0.06,0.43,"Canada, Ontario","Hard-Rock Emerging"),
    ("United Lithium Corp.","ULTH.CN","$",0.40,23,0.60,0.21,0.60,"Sweden","Hard-Rock Emerging"),
    ("Grid Metals","GRDM.V","$",0.14,23,-0.04,0.13,-0.10,"Canada, Manitoba","Hard-Rock Emerging"),
    ("Vanguard Mining Corp.","UUU.CN","$",0.43,23,1.77,1.87,1.46,"Argentina","Hard-Rock Emerging"),
    ("Odessa Minerals","ODE.AX","$",0.02,22,0.46,-0.46,0.27,"Australia, WA","Hard-Rock Emerging"),
    ("Pan American Energy Corp.","PNRG.CN","$",1.08,22,0.46,0.85,0.35,"Canada, Ontario","Hard-Rock Emerging"),
    ("Kali Metals","KM1.AX","$",0.21,21,0.14,0.37,0.17,"Australia, WA","Hard-Rock Emerging"),
    # ── SEDIMENTARY ───────────────────────────────────────────
    ("Lithium Americas","LAC","$",4.87,1478,0.12,-0.11,0.02,"USA, Nevada","Sedimentary"),
    ("American Battery Technology","ABAT","$",4.04,525,0.21,-0.21,0.09,"USA, Nevada","Sedimentary"),
    ("Ioneer","INR.AX","$",0.16,270,-0.18,-0.14,-0.23,"USA, Nevada","Sedimentary"),
    ("American Lithium","LI.V","$",0.85,159,0.33,0.15,0.23,"USA, Nevada","Sedimentary"),
    ("Surge Battery Metals","NILI.V","$",0.73,106,-0.01,0.52,-0.04,"USA, Nevada","Sedimentary"),
    ("Century Lithium","LCE.V","$",0.57,69,0.90,0.90,0.50,"USA, Nevada","Sedimentary"),
    ("Jindalee Resources","JRL.AX","$",0.61,40,0.15,0.11,0.09,"USA, Oregon","Sedimentary"),
    ("Nevada Lithium Resources","NVLH.V","$",0.17,32,-0.06,-0.17,0.00,"USA, Nevada","Sedimentary"),
    # ── DLE BRINE ─────────────────────────────────────────────
    ("Vulcan Energy Resources","VUL.AX","$",4.02,1254,-0.08,-0.36,-0.09,"Germany","DLE Brine"),
    ("Standard Lithium","SLI.V","$",5.90,1028,-0.04,0.10,-0.10,"USA, Arkansas","DLE Brine"),
    ("LibertyStream Infra Partners","LIB.V","$",1.39,211,0.11,0.11,0.11,"USA, Texas","DLE Brine"),
    ("Lake Resources","LKE.AX","$",0.09,143,-0.29,1.71,-0.32,"Argentina","DLE Brine"),
    ("E3 Lithium","ETL.V","$",1.21,77,0.36,0.16,0.30,"Canada, Alberta","DLE Brine"),
    ("Anson Resources","ASN.AX","$",0.06,66,-0.03,-0.28,-0.06,"USA, Utah","DLE Brine"),
    ("Stardust Power","SDST","$",4.06,40,0.33,-0.08,0.19,"USA, Oklahoma","DLE Brine"),
    ("International Battery Metals","IBAT.V","$",0.13,28,-0.07,-0.49,-0.16,"USA, Utah","DLE Brine"),
    ("Prairie Lithium","PL9.AX","$",0.01,27,-0.06,0.07,-0.06,"Canada, Saskatchewan","DLE Brine"),
    ("CleanTech Lithium","CTL.L","£",9.25,26,0.65,0.58,0.49,"Chile","DLE Brine"),
    ("LithiumBank","LBNK.V","$",0.50,23,-0.23,0.18,-0.21,"Canada, Alberta","DLE Brine"),
    # ── SALAR BRINE ───────────────────────────────────────────
    ("Galan Lithium","GLN.AX","$",0.38,282,0.15,1.59,0.12,"Argentina","Salar Brine"),
    ("Lithium Chile","LITH.V","$",0.57,93,0.00,0.25,-0.05,"Argentina","Salar Brine"),
    ("Argosy Minerals","AGY.AX","$",0.08,79,-0.29,0.50,-0.35,"Argentina","Salar Brine"),
    ("NOA Lithium Brines","NOAL.V","$",0.30,56,0.11,-0.02,0.11,"Argentina","Salar Brine"),
    ("Lithium South Development","LIS.V","$",0.42,39,-0.02,0.14,0.00,"Argentina","Salar Brine"),
    ("Lithium Energy Ltd","LEL.AX","$",0.37,27,0.00,0.00,0.00,"Argentina","Salar Brine"),
]

columns = ['company','ticker','currency','share_price','market_cap_usdm',
           'return_1m','return_3m','return_ytd','location','category']
df = pd.DataFrame(data, columns=columns)

for col in ['return_1m','return_3m','return_ytd']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['momentum_score'] = df['return_1m']*W_1M + df['return_3m']*W_3M + df['return_ytd']*W_YTD
df['market_cap_tier'] = pd.cut(df['market_cap_usdm'],
    bins=[0, TIER_MID, TIER_LARGE, TIER_MEGA, float('inf')],
    labels=['Small ($20M-$100M)','Mid ($100M-$1B)','Large ($1B-$10B)','Mega (>$10B)'])
df['accelerating'] = df['return_1m'] > (df['return_3m'] / 3)
df['all_positive'] = (df['return_1m']>0) & (df['return_3m']>0) & (df['return_ytd']>0)
df['all_negative'] = (df['return_1m']<0) & (df['return_3m']<0) & (df['return_ytd']<0)

print(f"✅ Loaded {len(df)} companies across {df['category'].nunique()} categories")
print(f"   Total sector market cap: ${df['market_cap_usdm'].sum():,.0f}M")


✅ Loaded 81 companies across 7 categories
   Total sector market cap: $154,213M


# 📡 Section 2: Live Data Enrichment via yfinance
Fetching fundamentals (P/E, P/B, margins, ROE, balance sheet) and 2-year price history.
This takes **3-5 minutes** depending on network speed.


In [3]:
# ── BATCH FETCH yfinance INFO FOR ALL TICKERS ────────────────
print("Fetching yfinance fundamentals for all tickers...")
print("=" * 60)

yf_data = {}
yf_success, yf_failed = [], []

for i, row in df.iterrows():
    ticker = row["ticker"]
    print(f"  [{i+1}/{len(df)}] {ticker:18s} ", end="", flush=True)
    info = fetch_yf_info(ticker, delay=0.25)
    if info:
        yf_data[ticker] = info
        yf_success.append(ticker)
        print("✅")
    else:
        yf_failed.append(ticker)
        print("❌")

print(f"\n{'='*60}")
print(f"✅ Success: {len(yf_success)}/{len(df)} ({len(yf_success)/len(df)*100:.0f}%)")
print(f"❌ Failed:  {len(yf_failed)}")


Fetching yfinance fundamentals for all tickers...
  [1/81] 000792.SZ          ✅
  [2/81] SQM                ✅
  [3/81] 1772.HK            ✅
  [4/81] ALB                ✅
  [5/81] 002466.SZ          ✅
  [6/81] 002240.SZ          ✅
  [7/81] 002497.SZ          ✅
  [8/81] 603876.SS          ✅
  [9/81] 002192.SZ          ✅
  [10/81] 002386.SZ          ✅
  [11/81] LAR                ✅
  [12/81] PLS.AX             ✅
  [13/81] MIN.AX             ✅
  [14/81] IGO.AX             ✅
  [15/81] LTR.AX             ✅
  [16/81] AMG.AS             ✅
  [17/81] SGML               ✅
  [18/81] ELV.AX             ✅
  [19/81] KOD.L              ✅
  [20/81] LIT                ✅
  [21/81] LIRC.TO            ✅
  [22/81] LITP               ✅
  [23/81] PMET.TO            ✅
  [24/81] CXO.AX             ✅
  [25/81] QTWO.V             ✅
  [26/81] WC8.AX             ✅
  [27/81] EUR.AX             ✅
  [28/81] LIFT.V             ✅
  [29/81] SAV.L              ✅
  [30/81] FL.V               ✅
  [31/81] ATLX               

In [4]:
# ── MERGE FUNDAMENTALS INTO MAIN DATAFRAME ───────────────────
fund_map = {
    "yf_market_cap": "marketCap", "enterprise_value": "enterpriseValue",
    "trailing_pe": "trailingPE", "forward_pe": "forwardPE",
    "price_to_book": "priceToBook", "price_to_sales": "priceToSalesTrailing12Months",
    "ev_to_ebitda": "enterpriseToEbitda", "ev_to_revenue": "enterpriseToRevenue",
    "profit_margin": "profitMargins", "operating_margin": "operatingMargins",
    "gross_margin": "grossMargins", "roe": "returnOnEquity", "roa": "returnOnAssets",
    "revenue": "totalRevenue", "ebitda": "ebitda",
    "total_cash": "totalCash", "total_debt": "totalDebt",
    "free_cash_flow": "freeCashflow",
    "revenue_growth": "revenueGrowth", "earnings_growth": "earningsGrowth",
    "current_ratio": "currentRatio", "debt_to_equity": "debtToEquity",
    "beta": "beta", "fifty_day_avg": "fiftyDayAverage",
    "two_hundred_day_avg": "twoHundredDayAverage",
    "52w_change": "52WeekChange", "dividend_yield": "dividendYield",
    "short_ratio": "shortRatio", "trailing_eps": "trailingEps",
    "shares_outstanding": "sharesOutstanding",
    "sector": "sector", "industry": "industry", "website": "website",
    "employees": "fullTimeEmployees",
}

for col_name, yf_key in fund_map.items():
    default = "N/A" if col_name in ("sector","industry","website") else np.nan
    vals = []
    for t in df["ticker"]:
        vals.append(safe_get(yf_data.get(t), yf_key, default))
    df[col_name] = vals

# Force numeric types on financial columns
numeric_cols = [c for c in fund_map.keys() if c not in ("sector","industry","website")]
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["net_debt"] = df["total_debt"].fillna(0) - df["total_cash"].fillna(0)
df["fcf_yield"] = np.where(
    (df["yf_market_cap"] > 0) & df["free_cash_flow"].notna(),
    df["free_cash_flow"] / df["yf_market_cap"], np.nan)

print(f"✅ Merged {len(fund_map)} fundamental columns")
print(f"   Companies with P/E data: {df['trailing_pe'].notna().sum()}")
print(f"   Companies with margin data: {df['profit_margin'].notna().sum()}")


✅ Merged 34 fundamental columns
   Companies with P/E data: 15
   Companies with margin data: 78


In [5]:
# ── FETCH 2-YEAR PRICE HISTORY FOR TOP 30 ────────────────────
print("Fetching 2-year price history for top 30 companies by market cap...")
top_tickers = df.nlargest(30, "market_cap_usdm")["ticker"].tolist()

price_histories = {}
for i, ticker in enumerate(top_tickers):
    print(f"  [{i+1}/{len(top_tickers)}] {ticker:18s} ", end="", flush=True)
    hist = fetch_yf_history(ticker, period="2y", delay=0.25)
    if hist is not None and len(hist) > 50:
        price_histories[ticker] = calculate_technical_indicators(hist)
        print(f"✅ ({len(hist)} days)")
    else:
        print("❌")

print(f"\n✅ Price history: {len(price_histories)}/{len(top_tickers)} companies")


Fetching 2-year price history for top 30 companies by market cap...
  [1/30] 000792.SZ          ✅ (484 days)
  [2/30] SQM                ✅ (501 days)
  [3/30] 1772.HK            ✅ (491 days)
  [4/30] ALB                ✅ (501 days)
  [5/30] 002466.SZ          ✅ (484 days)
  [6/30] PLS.AX             ✅ (507 days)
  [7/30] MIN.AX             ✅ (507 days)
  [8/30] 002240.SZ          ✅ (484 days)
  [9/30] IGO.AX             ✅ (507 days)
  [10/30] 002497.SZ          ✅ (484 days)
  [11/30] LTR.AX             ✅ (507 days)
  [12/30] 603876.SS          ✅ (484 days)
  [13/30] 002192.SZ          ✅ (484 days)
  [14/30] LIT                ✅ (501 days)
  [15/30] LAC                ✅ (501 days)
  [16/30] AMG.AS             ✅ (511 days)
  [17/30] VUL.AX             ✅ (507 days)
  [18/30] SGML               ✅ (501 days)
  [19/30] 002386.SZ          ✅ (484 days)
  [20/30] LAR                ✅ (501 days)
  [21/30] SLI.V              ✅ (502 days)
  [22/30] ELV.AX             ✅ (109 days)
  [23/30] PMET.TO

# 🌍 Section 3: Macro Context — Lithium Market, EV Demand & Rates
Data from FRED, IEA, Trading Economics, and analyst research (Feb 2026).


In [6]:
# ── LITHIUM PRICE HISTORY ─────────────────────────────────────
li_prices = pd.DataFrame({
    "year": [2018,2019,2020,2021,2022,2023,2024,2025,2026],
    "Li Carbonate (Asia, $/t)": [16000,9500,6500,18000,55000,25000,12000,11000,17380],
    "Li Hydroxide ($/t)":       [17000,10500,7000,20000,60000,28000,13000,11500,13360],
    "Spodumene 6% ($/t)":      [900,550,400,1200,6000,2500,1000,850,1100],
})
fig = go.Figure()
for col in li_prices.columns[1:]:
    fig.add_trace(go.Scatter(x=li_prices["year"], y=li_prices[col], name=col,
                             mode="lines+markers", line=dict(width=2.5)))
fig.update_layout(title="Lithium Commodity Prices (USD/tonne, annual avg/spot)",
                  yaxis_title="USD/tonne", template="plotly_white", height=420,
                  legend=dict(orientation="h", y=-0.15))
fig.show()

# ── EV SALES & DEMAND ───────────────────────────────────────
ev = pd.DataFrame({
    "year": [2019,2020,2021,2022,2023,2024,2025,2026],
    "Global EV Sales (M)": [2.1,3.0,6.6,10.5,14.2,17.0,18.5,22.0],
})
fig2 = px.bar(ev, x="year", y="Global EV Sales (M)",
              title="Global EV Sales (millions of units)",
              color_discrete_sequence=["#4CAF50"])
fig2.update_layout(template="plotly_white", height=350)
fig2.show()

# ── SUPPLY/DEMAND BALANCE ────────────────────────────────────
sd = pd.DataFrame({
    "year":[2021,2022,2023,2024,2025,2026,2027],
    "Supply (kt LCE)":[500,680,900,1050,1100,1200,1350],
    "Demand (kt LCE)":[450,720,880,980,1050,1250,1450],
})
sd["Balance"] = sd["Supply (kt LCE)"] - sd["Demand (kt LCE)"]
fig3 = make_subplots(specs=[[{"secondary_y": True}]])
fig3.add_trace(go.Bar(x=sd["year"], y=sd["Supply (kt LCE)"], name="Supply", marker_color="steelblue"), secondary_y=False)
fig3.add_trace(go.Bar(x=sd["year"], y=sd["Demand (kt LCE)"], name="Demand", marker_color="coral"), secondary_y=False)
fig3.add_trace(go.Scatter(x=sd["year"], y=sd["Balance"], name="Balance",
                          line=dict(color="red", width=2.5, dash="dot"), mode="lines+markers"), secondary_y=True)
fig3.update_layout(title="Lithium Supply-Demand Balance (kt LCE)", barmode="group",
                   template="plotly_white", height=400)
fig3.update_yaxes(title_text="kt LCE", secondary_y=False)
fig3.update_yaxes(title_text="Surplus / Deficit", secondary_y=True)
fig3.show()

print("📊 Key macro data points (Feb 2026):")
print("   Li₂CO₃ spot (NE Asia): ~$17,380/t (surging ~30% from Jan start)")
print("   Li₂CO₃ spot (Europe):  ~$13,360/t")
print("   GFEX Li futures (China): CNY 145,000-160,000/t")
print("   2026 supply-demand: deficit expected (-50 kt LCE)")
print("   Global EV sales 2026 forecast: ~22M+ units")
print("   Key drivers: China power storage spend, EV charging doubling, Jiangxi mine cancellations")


📊 Key macro data points (Feb 2026):
   Li₂CO₃ spot (NE Asia): ~$17,380/t (surging ~30% from Jan start)
   Li₂CO₃ spot (Europe):  ~$13,360/t
   GFEX Li futures (China): CNY 145,000-160,000/t
   2026 supply-demand: deficit expected (-50 kt LCE)
   Global EV sales 2026 forecast: ~22M+ units
   Key drivers: China power storage spend, EV charging doubling, Jiangxi mine cancellations


# 📊 Section 4: Sector Overview Dashboard

In [7]:
cat_summary = df.groupby('category').agg(
    count=('company','count'), total_mkt_cap=('market_cap_usdm','sum'),
    median_mkt_cap=('market_cap_usdm','median'), avg_ytd=('return_ytd','mean'),
    avg_1m=('return_1m','mean'),
).round(3).sort_values('total_mkt_cap', ascending=False)
print("📋 Category Summary:"); print(cat_summary.to_string())

fig = px.bar(cat_summary.reset_index().sort_values('total_mkt_cap'),
             x='total_mkt_cap', y='category', orientation='h', color='category',
             color_discrete_map=CATEGORY_COLORS, title='Total Market Cap by Category (USD M)')
fig.update_layout(template='plotly_white', showlegend=False, height=380)
fig.show()

fig2 = px.pie(cat_summary.reset_index(), values='total_mkt_cap', names='category',
              title='Market Cap Share', color='category', color_discrete_map=CATEGORY_COLORS)
fig2.update_traces(textposition='inside', textinfo='percent+label')
fig2.update_layout(height=450)
fig2.show()


📋 Category Summary:
                         count  total_mkt_cap  median_mkt_cap  avg_ytd  avg_1m
category                                                                      
Chemical Producers          11         114395          4542.0    0.069   0.105
Spodumene Producers          8          27514          2463.5    0.075   0.084
Hard-Rock Emerging          34           4202            78.0    0.210   0.243
DLE Brine                   11           2923            66.0    0.008   0.059
Sedimentary                  8           2679           132.5    0.082   0.182
Alt Investment Vehicles      3           1924           403.0    0.043   0.070
Salar Brine                  6            576            67.5   -0.028  -0.008


# 🌡️ Section 5: Performance Heatmap

In [8]:
top40 = df.nlargest(40, 'market_cap_usdm').sort_values('return_ytd', ascending=True)
heatmap_data = top40[['return_1m','return_3m','return_ytd']].values * 100
labels = [f"{t} ({c[:8]})" for t,c in zip(top40['ticker'], top40['category'])]
fig = go.Figure(data=go.Heatmap(
    z=heatmap_data, x=['1M %','3M %','YTD %'], y=labels,
    colorscale='RdYlGn', zmid=0,
    text=np.round(heatmap_data,1), texttemplate='%{text:.1f}%', textfont={"size":9},
    colorbar=dict(title='Return %')))
fig.update_layout(title='Performance Heatmap — Top 40 by Market Cap',
                  height=max(700,len(labels)*22), template='plotly_white', yaxis=dict(dtick=1))
fig.show()


# 📉 Section 6: Technical Analysis
Price charts with EMA 25/50/200, MACD, RSI, and Bollinger Bands.
Technical composite score: +1 per bullish signal, -1 per bearish (range: -5 to +5).


In [9]:
def plot_tech_chart(ticker, hdf):
    fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.03,
        row_heights=[0.45,0.15,0.20,0.20],
        subplot_titles=(f"{ticker} — Price & EMAs & BBands","Volume","MACD","RSI"))
    fig.add_trace(go.Candlestick(x=hdf.index, open=hdf["Open"], high=hdf["High"],
        low=hdf["Low"], close=hdf["Close"], name="OHLC",
        increasing_line_color="#4CAF50", decreasing_line_color="#F44336"), row=1, col=1)
    for ema,clr,nm in [("EMA_25","blue","EMA 25"),("EMA_50","orange","EMA 50"),("EMA_200","purple","EMA 200")]:
        if ema in hdf.columns:
            fig.add_trace(go.Scatter(x=hdf.index, y=hdf[ema], name=nm, line=dict(color=clr,width=1.2)), row=1, col=1)
    if "BB_Upper" in hdf.columns:
        fig.add_trace(go.Scatter(x=hdf.index, y=hdf["BB_Upper"], line=dict(color="grey",dash="dash",width=0.8), showlegend=False), row=1, col=1)
        fig.add_trace(go.Scatter(x=hdf.index, y=hdf["BB_Lower"], fill="tonexty", fillcolor="rgba(128,128,128,0.08)",
            line=dict(color="grey",dash="dash",width=0.8), showlegend=False), row=1, col=1)
    vc = ["#4CAF50" if c>=o else "#F44336" for c,o in zip(hdf["Close"],hdf["Open"])]
    fig.add_trace(go.Bar(x=hdf.index, y=hdf["Volume"], marker_color=vc, showlegend=False), row=2, col=1)
    if "MACD" in hdf.columns:
        fig.add_trace(go.Scatter(x=hdf.index, y=hdf["MACD"], name="MACD", line=dict(color="blue",width=1.3)), row=3, col=1)
        fig.add_trace(go.Scatter(x=hdf.index, y=hdf["MACD_Signal"], name="Signal", line=dict(color="red",width=1.3)), row=3, col=1)
        mc = ["#4CAF50" if v>=0 else "#F44336" for v in hdf["MACD_Hist"]]
        fig.add_trace(go.Bar(x=hdf.index, y=hdf["MACD_Hist"], marker_color=mc, showlegend=False), row=3, col=1)
    if "RSI" in hdf.columns:
        fig.add_trace(go.Scatter(x=hdf.index, y=hdf["RSI"], name="RSI", line=dict(color="purple",width=1.3)), row=4, col=1)
        fig.add_hline(y=70, line_dash="dash", line_color="red", row=4, col=1)
        fig.add_hline(y=30, line_dash="dash", line_color="green", row=4, col=1)
    fig.update_layout(height=850, template="plotly_white", xaxis_rangeslider_visible=False,
                      legend=dict(orientation="h", y=1.02))
    fig.update_yaxes(range=[0,100], row=4, col=1)
    fig.show()

tickers_w_hist = [t for t in df.nlargest(20,"market_cap_usdm")["ticker"] if t in price_histories]
print(f"📈 Technical charts for {len(tickers_w_hist)} companies:\n")
for tk in tickers_w_hist[:12]:
    plot_tech_chart(tk, price_histories[tk])


📈 Technical charts for 20 companies:



In [10]:
# ── TECHNICAL SIGNALS SUMMARY ─────────────────────────────────
tech_signals = []
for ticker, hdf in price_histories.items():
    if len(hdf) > 50:
        sigs = get_technical_signal(hdf.iloc[-1])
        sigs["ticker"] = ticker
        sigs["close"] = round(hdf.iloc[-1]["Close"], 4)
        cr = df[df["ticker"]==ticker]
        sigs["company"] = cr["company"].values[0] if len(cr)>0 else ticker
        sigs["category"] = cr["category"].values[0] if len(cr)>0 else "N/A"
        tech_signals.append(sigs)

df_tech = pd.DataFrame(tech_signals)
if len(df_tech) > 0:
    df_tech = df_tech.sort_values("tech_score", ascending=False)
    print("📊 Technical Signals Summary:")
    print(df_tech[["ticker","company","close","trend","ema_cross","rsi","rsi_value","macd","bollinger","tech_score"]].to_string(index=False))

    dts = df_tech.sort_values("tech_score", ascending=True)
    fig = go.Figure(go.Bar(x=dts["tech_score"], y=dts["ticker"], orientation="h",
        marker_color=["#4CAF50" if s>0 else "#F44336" if s<0 else "#999" for s in dts["tech_score"]],
        text=dts["tech_score"], textposition="outside"))
    fig.update_layout(title="Technical Composite Score (-5 to +5)", template="plotly_white",
                      height=max(400,len(dts)*25))
    fig.show()
    tech_map = dict(zip(df_tech["ticker"], df_tech["tech_score"]))
    df["tech_score"] = df["ticker"].map(tech_map)
else:
    df["tech_score"] = np.nan


📊 Technical Signals Summary:
   ticker                             company   close   trend    ema_cross        rsi  rsi_value    macd  bollinger  tech_score
  1772.HK                             Ganfeng  67.500 Bullish Golden Cross    Neutral       64.3 Bullish   Mid-Band           3
002192.SZ                              YOUNGY  56.260 Bullish Golden Cross    Neutral       42.8 Bullish   Mid-Band           3
     SGML                       Sigma Lithium  12.600 Bullish Golden Cross    Neutral       53.7 Bullish   Mid-Band           3
    SLI.V                    Standard Lithium   6.510 Bullish Golden Cross    Neutral       51.6 Bullish   Mid-Band           3
      LAC                    Lithium Americas   5.040 Bullish Golden Cross    Neutral       47.3 Bullish   Mid-Band           3
      ALB                           Albemarle 186.830 Bullish Golden Cross    Neutral       59.7 Bullish Near Upper           2
      LIT Global X Lithium & Battery Tech ETF  75.590 Bullish Golden Cross 

# 📊 Section 7: Fundamental Analysis
Valuation multiples, profitability, balance sheet health, cash flow, and growth from yfinance.


In [11]:
# ── VALUATION MULTIPLES ───────────────────────────────────────
val_df = df[["ticker","company","category","market_cap_usdm","trailing_pe","forward_pe",
             "price_to_book","price_to_sales","ev_to_ebitda","ev_to_revenue"]].dropna(subset=["trailing_pe"])
if len(val_df) > 0:
    print(f"📊 Valuation Multiples ({len(val_df)} companies):")
    print(val_df.sort_values("ev_to_ebitda").to_string(index=False))
    pe_v = val_df[(val_df["trailing_pe"]>0)&(val_df["trailing_pe"]<100)]
    if len(pe_v)>2:
        fig = px.histogram(pe_v, x="trailing_pe", nbins=20, color="category",
                           color_discrete_map=CATEGORY_COLORS, title="Trailing P/E Distribution")
        fig.update_layout(template="plotly_white", height=380); fig.show()
    ev_v = val_df[(val_df["ev_to_ebitda"]>0)&(val_df["ev_to_ebitda"]<50)]
    if len(ev_v)>2:
        fig2 = px.scatter(ev_v, x="market_cap_usdm", y="ev_to_ebitda", color="category",
                          hover_name="ticker", log_x=True, size="market_cap_usdm", size_max=30,
                          color_discrete_map=CATEGORY_COLORS, title="EV/EBITDA vs Market Cap")
        fig2.update_layout(template="plotly_white", height=420); fig2.show()
else:
    print("⚠️ No valuation data available.")


📊 Valuation Multiples (15 companies):
   ticker                             company                category  market_cap_usdm  trailing_pe  forward_pe  price_to_book  price_to_sales  ev_to_ebitda  ev_to_revenue
    SLI.V                    Standard Lithium               DLE Brine             1028     7.843374  -49.694660       3.886837             NaN      -110.271            NaN
   AGY.AX                     Argosy Minerals             Salar Brine               79     2.150000   -0.122857       1.686275             NaN       -60.364            NaN
   LITH.V                       Lithium Chile             Salar Brine               93          inf         NaN       3.315218             NaN       -47.826            NaN
    CRE.V              Critical Elements Corp      Hard-Rock Emerging               80          inf   -2.718750       1.283186             NaN       -15.624            NaN
   PL9.AX                     Prairie Lithium               DLE Brine               27          inf   

In [12]:
# ── PROFITABILITY ─────────────────────────────────────────────
prof = df[["ticker","company","category","market_cap_usdm","gross_margin","operating_margin",
           "profit_margin","roe","roa"]].dropna(subset=["profit_margin"])
if len(prof) > 0:
    top_prof = prof.nlargest(20, "market_cap_usdm")
    fig = go.Figure()
    for m,c in [("gross_margin","#4CAF50"),("operating_margin","#2196F3"),("profit_margin","#FF9800")]:
        fig.add_trace(go.Bar(name=m.replace("_"," ").title(), x=top_prof["ticker"], y=top_prof[m]*100, marker_color=c))
    fig.update_layout(title="Profitability Margins — Top 20", barmode="group",
                      template="plotly_white", height=420, yaxis_title="Margin %", xaxis_tickangle=-45)
    fig.show()
    roe_v = prof[prof["roe"].notna()].sort_values("roe", ascending=True)
    if len(roe_v) > 2:
        fig2 = go.Figure(go.Bar(x=roe_v["roe"]*100, y=roe_v["ticker"], orientation="h",
            marker_color=["#4CAF50" if v>0 else "#F44336" for v in roe_v["roe"]],
            text=[f"{v*100:.1f}%" for v in roe_v["roe"]], textposition="outside"))
        fig2.update_layout(title="Return on Equity (%)", template="plotly_white",
                           height=max(400,len(roe_v)*22))
        fig2.show()


In [13]:
# ── BALANCE SHEET HEALTH ──────────────────────────────────────
bs = df[["ticker","company","category","market_cap_usdm","current_ratio","debt_to_equity",
         "total_cash","total_debt","net_debt"]].dropna(subset=["current_ratio"])
if len(bs) > 0:
    cr_v = bs.nlargest(25,"market_cap_usdm").sort_values("current_ratio",ascending=True)
    fig = go.Figure(go.Bar(x=cr_v["current_ratio"], y=cr_v["ticker"], orientation="h",
        marker_color=["#4CAF50" if v>=1.5 else "#FF9800" if v>=1.0 else "#F44336" for v in cr_v["current_ratio"]],
        text=[f"{v:.2f}x" for v in cr_v["current_ratio"]], textposition="outside"))
    fig.add_vline(x=1.5, line_dash="dash", line_color="green", annotation_text="Healthy")
    fig.add_vline(x=1.0, line_dash="dash", line_color="red", annotation_text="Min")
    fig.update_layout(title="Current Ratio (Liquidity)", template="plotly_white",
                      height=max(400,len(cr_v)*22))
    fig.show()


In [14]:
# ── GROWTH & CASH FLOW ────────────────────────────────────────
gdf = df[["ticker","company","category","revenue_growth","earnings_growth","fcf_yield"]].dropna(subset=["revenue_growth"])
if len(gdf) > 0:
    gdf_s = gdf.sort_values("revenue_growth")
    fig = go.Figure(go.Bar(x=gdf_s["revenue_growth"]*100, y=gdf_s["ticker"], orientation="h",
        marker_color=["#4CAF50" if v>0 else "#F44336" for v in gdf_s["revenue_growth"]],
        text=[f"{v*100:.1f}%" for v in gdf_s["revenue_growth"]], textposition="outside"))
    fig.update_layout(title="Revenue Growth YoY (%)", template="plotly_white",
                      height=max(400,len(gdf_s)*22))
    fig.show()


In [15]:
# ── FUNDAMENTAL COMPOSITE SCORE (0-10) ────────────────────────
def calc_fund_score(row):
    s = 0
    if pd.notna(row.get("profit_margin")) and row["profit_margin"] > 0: s += 1
    if pd.notna(row.get("operating_margin")) and row["operating_margin"] > 0.1: s += 1
    if pd.notna(row.get("roe")) and row["roe"] > 0.1: s += 1
    if pd.notna(row.get("trailing_pe")) and 0 < row["trailing_pe"] < 25: s += 1
    if pd.notna(row.get("ev_to_ebitda")) and 0 < row["ev_to_ebitda"] < 15: s += 1
    if pd.notna(row.get("current_ratio")) and row["current_ratio"] > 1.5: s += 1
    if pd.notna(row.get("debt_to_equity")) and row["debt_to_equity"] < 100: s += 1
    if pd.notna(row.get("revenue_growth")) and row["revenue_growth"] > 0: s += 1
    if pd.notna(row.get("earnings_growth")) and row["earnings_growth"] > 0: s += 1
    if pd.notna(row.get("fcf_yield")) and row["fcf_yield"] > 0: s += 1
    return s

df["fundamental_score"] = df.apply(calc_fund_score, axis=1)
has_data = df["trailing_pe"].notna() | df["profit_margin"].notna() | df["current_ratio"].notna()
df_fr = df[has_data].sort_values("fundamental_score", ascending=False)
if len(df_fr) > 0:
    print(f"📊 Fundamental Score (0-10, {len(df_fr)} companies):")
    print(df_fr[["ticker","company","fundamental_score","trailing_pe","profit_margin","roe","current_ratio","revenue_growth"]].head(20).to_string(index=False))
    top_f = df_fr.head(25).sort_values("fundamental_score", ascending=True)
    fig = go.Figure(go.Bar(x=top_f["fundamental_score"], y=top_f["ticker"], orientation="h",
        marker_color=[CATEGORY_COLORS.get(c,"#999") for c in top_f["category"]],
        text=top_f["fundamental_score"], textposition="outside"))
    fig.update_layout(title="Fundamental Quality Score (0-10)", template="plotly_white",
                      height=max(400,len(top_f)*22))
    fig.show()


📊 Fundamental Score (0-10, 80 companies):
   ticker                     company  fundamental_score  trailing_pe  profit_margin      roe  current_ratio  revenue_growth
000792.SZ  Qinghai Salt Lake Industry                  8    31.754387        0.38140  0.15503          8.216           0.348
002192.SZ                      YOUNGY                  7    82.735290        0.26444  0.04787          2.737           0.347
002497.SZ               Sichuan Yahua                  6    76.263160        0.05569  0.03506          3.020           0.320
      SQM                         SQM                  6          NaN        0.12124  0.09955          2.823           0.089
   MIN.AX           Mineral Resources                  6    28.777230        0.07642  0.12105          1.564           0.333
   PL9.AX             Prairie Lithium                  6          inf        0.74338  0.17050          2.891             NaN
   AMG.AS       AMG Critical Minerals                  5    39.617023        0.0227

# 📈 Section 8: Return Distributions & Top/Bottom Performers

In [16]:
# YTD histogram
fig = px.histogram(df, x=df['return_ytd']*100, nbins=40, title='YTD Return Distribution',
                   labels={'x':'YTD Return %'}, color_discrete_sequence=['steelblue'])
fig.add_vline(x=df['return_ytd'].median()*100, line_dash='dash', line_color='red',
              annotation_text=f"Median: {df['return_ytd'].median()*100:.1f}%")
fig.add_vline(x=0, line_dash='solid', line_color='black', line_width=1)
fig.update_layout(template='plotly_white', height=380); fig.show()

# Box plots
for period,col in [('1M','return_1m'),('3M','return_3m'),('YTD','return_ytd')]:
    fig = px.box(df, x='category', y=df[col]*100, color='category',
                 color_discrete_map=CATEGORY_COLORS, title=f'{period} Return by Category (%)', points='all')
    fig.update_layout(template='plotly_white', showlegend=False, xaxis_tickangle=-25, height=420); fig.show()


In [17]:
# Top/Bottom performers
def perf_bar(dfi, col, title, n=10, asc=False):
    sub = dfi.nsmallest(n,col) if asc else dfi.nlargest(n,col)
    sub = sub.sort_values(col, ascending=True)
    fig = go.Figure(go.Bar(x=sub[col]*100, y=sub['ticker'], orientation='h',
        marker_color=['#4CAF50' if v>=0 else '#F44336' for v in sub[col]],
        text=[f"{v*100:.1f}%" for v in sub[col]], textposition='outside'))
    fig.update_layout(title=title, template='plotly_white', height=380); fig.show()

perf_bar(df,'return_ytd','🟢 Top 10 YTD',10)
perf_bar(df,'return_ytd','🔴 Bottom 10 YTD',10,True)
perf_bar(df,'return_1m','🟢 Top 10 1-Month',10)
perf_bar(df,'return_1m','🔴 Bottom 10 1-Month',10,True)

print(f"\n✅ Consistent Winners: {df['all_positive'].sum()}")
print(df[df['all_positive']][['company','ticker','category','market_cap_usdm','return_1m','return_3m','return_ytd']].sort_values('return_ytd',ascending=False).to_string(index=False))
print(f"\n❌ Consistent Losers: {df['all_negative'].sum()}")
print(df[df['all_negative']][['company','ticker','category','market_cap_usdm','return_1m','return_3m','return_ytd']].sort_values('return_ytd').to_string(index=False))



✅ Consistent Winners: 43
                            company    ticker                category  market_cap_usdm  return_1m  return_3m  return_ytd
              Vanguard Mining Corp.    UUU.CN      Hard-Rock Emerging               23       1.77       1.87        1.46
                   MAX Power Mining     MAXXF      Hard-Rock Emerging               76       0.72       0.47        0.73
               United Lithium Corp.   ULTH.CN      Hard-Rock Emerging               23       0.60       0.21        0.60
                     Kodal Minerals     KOD.L     Spodumene Producers              133       0.48       0.63        0.58
                   European Lithium    EUR.AX      Hard-Rock Emerging              267       0.47       0.15        0.52
                    Century Lithium     LCE.V             Sedimentary               69       0.90       0.90        0.50
                  CleanTech Lithium     CTL.L               DLE Brine               26       0.65       0.58        0.49
      

# 💰 Section 9: Market Cap & Geographic Analysis

In [18]:
# Scatter: mkt cap vs YTD
fig = px.scatter(df, x='market_cap_usdm', y=df['return_ytd']*100, color='category',
                 size='market_cap_usdm', hover_name='ticker', log_x=True, size_max=40,
                 color_discrete_map=CATEGORY_COLORS, title='Market Cap vs YTD Return')
fig.add_hline(y=0, line_dash='dash', line_color='grey')
fig.update_layout(template='plotly_white', height=500); fig.show()

# Bubble: 1m vs 3m
fig2 = px.scatter(df, x=df['return_1m']*100, y=df['return_3m']*100, size='market_cap_usdm',
                  color='category', hover_name='ticker', size_max=50,
                  color_discrete_map=CATEGORY_COLORS, title='1M vs 3M Returns (size=mkt cap)')
fig2.add_hline(y=0, line_dash='dash', line_color='grey')
fig2.add_vline(x=0, line_dash='dash', line_color='grey')
fig2.update_layout(template='plotly_white', height=500); fig2.show()

# Treemap
fig3 = px.treemap(df, path=['location','ticker'], values='market_cap_usdm',
                  color='return_ytd', color_continuous_scale='RdYlGn', color_continuous_midpoint=0,
                  title='Market Cap Treemap by Location (colour = YTD)')
fig3.update_layout(height=550); fig3.show()


# 🚀 Section 10: Momentum & Trend Analysis

In [19]:
mr = df.sort_values('momentum_score', ascending=False).head(20).sort_values('momentum_score', ascending=True)
fig = go.Figure(go.Bar(x=mr['momentum_score']*100, y=mr['ticker'], orientation='h',
    marker_color=[CATEGORY_COLORS.get(c,'#999') for c in mr['category']],
    text=[f"{v*100:.1f}%" for v in mr['momentum_score']], textposition='outside'))
fig.update_layout(title='Top 20 Momentum Score (40% 1M + 35% 3M + 25% YTD)',
                  template='plotly_white', height=550); fig.show()

cat_mom = df.groupby('category')['momentum_score'].mean().sort_values()
fig2 = go.Figure(go.Bar(x=cat_mom.values*100, y=cat_mom.index, orientation='h',
    marker_color=[CATEGORY_COLORS.get(c,'#999') for c in cat_mom.index],
    text=[f"{v:.1f}%" for v in cat_mom.values*100], textposition='outside'))
fig2.update_layout(title='Avg Momentum by Category', template='plotly_white', height=350); fig2.show()


# 🔬 Section 11: Correlation & Factor Analysis

In [20]:
corr_cols = ['market_cap_usdm','return_1m','return_3m','return_ytd','momentum_score']
corr_m = df[corr_cols].corr()
corr_m.index = ['Mkt Cap','1M','3M','YTD','Momentum']
corr_m.columns = corr_m.index
fig = px.imshow(corr_m.round(2), text_auto=True, color_continuous_scale='RdBu_r',
                title='Correlation Matrix')
fig.update_layout(template='plotly_white', height=420, width=500); fig.show()


# 🎯 Section 12: Screening & Signal Dashboard
Seven screens including technical + fundamental combo signals. Combined score
weights: 30% Technical + 40% Fundamental + 30% Momentum.


In [21]:
# Screens
df['screen_value_momentum'] = (df['return_ytd']>0) & (df['market_cap_usdm']<500)
df['screen_largecap_strength'] = (df['market_cap_usdm']>1000) & (df['return_1m']>0)
df['screen_turnaround'] = (df['return_ytd']<0) & (df['return_1m']>0)
df['screen_avoid'] = df['all_negative']
df['screen_consistent'] = df['all_positive']
df['screen_tech_oversold_quality'] = (df['tech_score'].fillna(0)<=-2) & (df['fundamental_score']>=5)
df['screen_high_quality_momentum'] = (df['fundamental_score']>=6) & (df['momentum_score']>0) & (df['tech_score'].fillna(0)>=1)

# Combined score
for col in ['tech_score','fundamental_score','momentum_score']:
    cc = df[col].fillna(0)
    cr = cc.max() - cc.min()
    df[f"{col}_norm"] = (cc - cc.min()) / cr if cr > 0 else 0

df['combined_score'] = df['tech_score_norm']*0.30 + df['fundamental_score_norm']*0.40 + df['momentum_score_norm']*0.30

screens = {
    "🟢 Value Momentum (YTD+ & <$500M)": "screen_value_momentum",
    "🔵 Large Cap Strength (>$1B & 1M+)": "screen_largecap_strength",
    "🟡 Turnaround (YTD- but 1M+)": "screen_turnaround",
    "🔴 Avoid (all negative)": "screen_avoid",
    "🏆 Consistent Winners (all positive)": "screen_consistent",
    "🔬 Tech Oversold + Quality Fundamentals": "screen_tech_oversold_quality",
    "⭐ High Quality + Momentum + Tech": "screen_high_quality_momentum",
}

for label, col in screens.items():
    sub = df[df[col]].sort_values('combined_score', ascending=False)
    print(f"\n{'='*70}\n{label}: {len(sub)} companies\n{'='*70}")
    if len(sub) > 0:
        print(sub[['company','ticker','category','market_cap_usdm','return_ytd',
                    'tech_score','fundamental_score','combined_score']].head(15).to_string(index=False))

# Top 20 combined
print(f"\n{'='*70}\n⭐ TOP 20 COMBINED SCORE (30% Tech + 40% Fundamental + 30% Momentum)\n{'='*70}")
tc = df.sort_values('combined_score', ascending=False).head(20)
print(tc[['company','ticker','category','market_cap_usdm','tech_score','fundamental_score',
          'momentum_score','combined_score']].to_string(index=False))

tcs = tc.sort_values('combined_score', ascending=True)
fig = go.Figure(go.Bar(x=tcs['combined_score'], y=tcs['ticker'], orientation='h',
    marker_color=[CATEGORY_COLORS.get(c,'#999') for c in tcs['category']],
    text=[f"{v:.3f}" for v in tcs['combined_score']], textposition='outside'))
fig.update_layout(title='Combined Score — Top 20', template='plotly_white', height=550)
fig.show()



🟢 Value Momentum (YTD+ & <$500M): 35 companies
                  company  ticker                category  market_cap_usdm  return_ytd  tech_score  fundamental_score  combined_score
    Vanguard Mining Corp.  UUU.CN      Hard-Rock Emerging               23        1.46         NaN                  3        0.570000
            Galan Lithium  GLN.AX             Salar Brine              282        0.12         2.0                  1        0.425130
     Lithium Royalty Corp LIRC.TO Alt Investment Vehicles              403        0.04         0.0                  4        0.383066
          Century Lithium   LCE.V             Sedimentary               69        0.50         NaN                  2        0.378541
     United Lithium Corp. ULTH.CN      Hard-Rock Emerging               23        0.60         NaN                  3        0.377219
          Q2 Metals Corp.  QTWO.V      Hard-Rock Emerging              329        0.17         1.0                  1        0.363681
         Europ

# 📄 Section 13: SEC EDGAR & Official Filings
Analyst recommendations, insider trades, and earnings dates for US-listed companies.
Direct links to SEC EDGAR for 10-K/10-Q filings.


In [22]:
# US-listed tickers (no exchange suffix)
us_tickers = [t for t in df[~df['ticker'].str.contains(r'\.',na=False)]['ticker'] if t in yf_data]

for ticker in us_tickers[:8]:
    print(f"\n{'='*50}\n  {ticker}\n{'='*50}")
    try:
        stk = yf.Ticker(ticker)
        try:
            recs = stk.recommendations
            if recs is not None and len(recs) > 0:
                print("  📋 Analyst Recommendations (recent):")
                print(recs.tail(5).to_string())
        except: pass
        try:
            ins = stk.insider_trades
            if ins is not None and len(ins) > 0:
                print("  👤 Insider Trades (recent):")
                print(ins.head(5).to_string())
        except: pass
        try:
            earn = stk.earnings_dates
            if earn is not None and len(earn) > 0:
                print("  📅 Earnings Dates:")
                print(earn.head(3).to_string())
        except: pass
        time.sleep(0.5)
    except Exception as e:
        print(f"  ⚠️ {e}")

# SEC EDGAR direct links
print("\n\n📄 SEC EDGAR Filing Links:")
sec_ciks = {"ALB":"0001178670","SQM":"0000914208","LAC":"0001701903","LIT":"0001364382",
            "ABAT":"0001783940","SGML":"0001911516","ATLX":"0001826397","SDST":"0001862150"}
for t, cik in sec_ciks.items():
    if t in df['ticker'].values:
        print(f"  {t}: https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={cik}&type=10-K&dateb=&owner=include&count=5")



  SQM
  📋 Analyst Recommendations (recent):
  period  strongBuy  buy  hold  sell  strongSell
0     0m          4    4     8     0           1
1    -1m          4    4     8     0           1
2    -2m          4    3     9     0           1
3    -3m          3    5     8     0           1
  📅 Earnings Dates:
                           EPS Estimate  Reported EPS  Surprise(%)
Earnings Date                                                     
2026-02-27 16:00:00-05:00          0.80           NaN          NaN
2025-11-19 04:00:00-05:00          0.66          0.62        -4.68
2025-08-20 04:00:00-04:00          0.54          0.31       -42.12

  ALB
  📋 Analyst Recommendations (recent):
  period  strongBuy  buy  hold  sell  strongSell
0     0m          2   11    12     0           0
1    -1m          2   10    14     0           0
2    -2m          1    7    18     0           0
3    -3m          1    6    17     2           0
  📅 Earnings Dates:
                           EPS Estimate  Repo

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: LIT"}}}
LIT: No earnings dates found, symbol may be delisted



  LITP


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: LITP"}}}
LITP: No earnings dates found, symbol may be delisted



  ATLX
  📋 Analyst Recommendations (recent):
  period  strongBuy  buy  hold  sell  strongSell
0     0m          0    2     0     0           0
1    -1m          0    2     0     0           0
  📅 Earnings Dates:
                           EPS Estimate  Reported EPS  Surprise(%)
Earnings Date                                                     
2025-11-13 17:00:00-05:00         -0.64         -0.35        45.31
2025-08-04 08:00:00-04:00         -0.59         -0.31        47.46
2025-05-09 16:00:00-04:00         -0.50         -0.55       -10.00

  MAXXF


MAXXF: No earnings dates found, symbol may be delisted




📄 SEC EDGAR Filing Links:
  ALB: https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=0001178670&type=10-K&dateb=&owner=include&count=5
  SQM: https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=0000914208&type=10-K&dateb=&owner=include&count=5
  LAC: https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=0001701903&type=10-K&dateb=&owner=include&count=5
  LIT: https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=0001364382&type=10-K&dateb=&owner=include&count=5
  ABAT: https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=0001783940&type=10-K&dateb=&owner=include&count=5
  SGML: https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=0001911516&type=10-K&dateb=&owner=include&count=5
  ATLX: https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=0001826397&type=10-K&dateb=&owner=include&count=5
  SDST: https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=0001862150&type=10-K&dateb=&owner=include&count=5


# 📝 Section 14: Executive Summary

In [23]:
total = len(df)
mkt = df['market_cap_usdm'].sum()
pct_ytd = (df['return_ytd']>0).mean()*100
pct_1m = (df['return_1m']>0).mean()*100
best_cat = df.groupby('category')['return_ytd'].mean().idxmax()
best_ytd = df.groupby('category')['return_ytd'].mean().max()*100
top5 = df.nlargest(5, 'combined_score')

print("="*70)
print(f"  LITHIUM SECTOR SCOREBOARD — EXECUTIVE SUMMARY")
print(f"  {SCOREBOARD_DATE}")
print("="*70)
print(f"\n📊 UNIVERSE: {total} companies | ${mkt:,.0f}M total market cap")
print(f"📈 BREADTH: {pct_ytd:.0f}% positive YTD | {pct_1m:.0f}% positive 1M")
print(f"🏆 BEST CATEGORY: {best_cat} ({best_ytd:+.1f}% avg YTD)")
print(f"\n📡 DATA QUALITY:")
print(f"   yfinance fundamentals: {len(yf_success)}/{total} ({len(yf_success)/total*100:.0f}%)")
print(f"   Price histories (2yr): {len(price_histories)} companies")
print(f"   With P/E data: {df['trailing_pe'].notna().sum()}")
print(f"   With margin data: {df['profit_margin'].notna().sum()}")
print(f"\n⭐ TOP 5 BY COMBINED SCORE (Tech + Fundamental + Momentum):")
for _, r in top5.iterrows():
    print(f"   {r['ticker']:12s} {r['company']:40s} [Score: {r['combined_score']:.3f}]")
    print(f"              Tech={r['tech_score'] if pd.notna(r['tech_score']) else 'N/A'} | Fund={r['fundamental_score']:.0f} | Mom={r['momentum_score']*100:.1f}%")
print(f"\n🌍 MACRO: Li₂CO₃ ~$17,380/t (NE Asia) | 2026 deficit expected | EV sales ~22M forecast")
print(f"\n⚠️  NOT INVESTMENT ADVICE. DO YOUR OWN RESEARCH.")
print("="*70)


  LITHIUM SECTOR SCOREBOARD — EXECUTIVE SUMMARY
  January 31, 2026

📊 UNIVERSE: 81 companies | $154,213M total market cap
📈 BREADTH: 62% positive YTD | 68% positive 1M
🏆 BEST CATEGORY: Hard-Rock Emerging (+21.0% avg YTD)

📡 DATA QUALITY:
   yfinance fundamentals: 80/81 (99%)
   Price histories (2yr): 30 companies
   With P/E data: 15
   With margin data: 78

⭐ TOP 5 BY COMBINED SCORE (Tech + Fundamental + Momentum):
   000792.SZ    Qinghai Salt Lake Industry               [Score: 0.706]
              Tech=2.0 | Fund=8 | Mom=19.2%
   002192.SZ    YOUNGY                                   [Score: 0.697]
              Tech=3.0 | Fund=7 | Mom=6.6%
   MIN.AX       Mineral Resources                        [Score: 0.591]
              Tech=2.0 | Fund=6 | Mom=9.4%
   002497.SZ    Sichuan Yahua                            [Score: 0.591]
              Tech=2.0 | Fund=6 | Mom=9.3%
   UUU.CN       Vanguard Mining Corp.                    [Score: 0.570]
              Tech=N/A | Fund=3 | Mom=172.8%

🌍